# Conditional Distributions

Marginals *forget* one variable; **conditional distributions** do the opposite — they *fix* one variable at a known value and ask how the other one behaves. Only care about the 9-year-olds? Take that **slice** of the joint table (one row), then **normalize** it so it sums to 1. The result, $P_{Y \mid X=x}(y) = \frac{P_{XY}(x,y)}{P_X(x)}$, is just the conditional-probability rule from Lesson 1 wearing distribution clothes — and the same recipe works for continuous densities, slicing the call-center mountain with a plane. It is based on the fifth video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

With marginals we summarized one variable while ignoring the other. Now we want something different: **observe one variable given that the value of the other one is known**. What is the height distribution *of the 9-year-olds*? What ratings do customers give *after waiting exactly four minutes*? These are everyday questions — every "among users who…", "for patients with…", "given that…" in data science is a conditional distribution.

After completing this notebook you will be able to:

- Extract a conditional distribution by **slicing** the joint table and **normalizing** the slice.
- Use the formula $P_{Y \mid X=x}(y) = \frac{P_{XY}(x, y)}{P_X(x)}$ and derive it from the conditional-probability rule.
- Check conditionals against intuition on independent dice.
- Extend to continuous variables: $f_{Y \mid X=x}(y) = \frac{f_{XY}(x, y)}{f_X(x)}$ — slicing a density surface.

## 2. Theory

### Take a slice…

Back to the children's joint table:

| $P_{XY}$ | 45 | 46 | 47 | 48 | 49 | 50 |
|---|---|---|---|---|---|---|
| **7** | 0.1 | 0.2 | 0 | 0 | 0 | 0 |
| **8** | 0 | 0 | 0.2 | 0 | 0 | 0 |
| **9** | 0 | 0 | 0 | 0 | **0.3** | **0.1** |
| **10** | 0 | 0 | 0 | 0 | 0 | 0.1 |

Suppose age 9 is the only age we care about, and we want the distribution of height among those children. Fix $X = 9$: of the whole table we only look at **that row** — the slice $(0, 0, 0, 0, 0.3, 0.1)$.

### …then normalize

A small caveat: a probability distribution must add to 1, and this row adds to $0.3 + 0.1 = \tfrac{4}{10}$. The fix: **divide everything by the row sum**:

$$
P_{Y \mid X=9}(49) = \frac{0.3}{0.4} = \frac34
\qquad
P_{Y \mid X=9}(50) = \frac{0.1}{0.4} = \frac14
$$

Three quarters of the 9-year-olds are 49 inches tall; one quarter are 50.

### Why dividing by the row sum is the conditional-probability rule

Recall from Lesson 1: $P(A \text{ and } B) = P(A)\,P(B \mid A)$. With $A$ = "$X = 9$" and $B$ = "$Y = 49$":

$$
P(Y = 49 \mid X = 9) = \frac{P(X = 9,\ Y = 49)}{P(X = 9)} = \frac{3/10}{4/10} = \frac34
$$

The row sum *is* the marginal $P_X(9)$ — normalizing a slice is exactly applying the conditional-probability rule. The general formula, worth remembering:

$$
P_{Y \mid X=x}(y) = \frac{P_{XY}(x, y)}{P_X(x)}
$$

joint on top, **marginal of the fixed variable** on the bottom.

### Sanity check: independent dice

$X$ = first die, $Y$ = second. What is $P(Y = 1 \mid X = 4)$? Take row $X = 4$ (six cells of $\tfrac1{36}$), normalize by the row sum $\tfrac16$: every cell becomes $\tfrac{1/36}{1/6} = \tfrac16$. Knowing the first die tells you nothing about the second — the conditional equals the plain marginal, which is precisely what independence means.

### Continuous conditionals: slicing the mountain

For the call-center density (waiting time $X$, rating $Y$): to find the distribution of ratings *given a waiting time of 4 minutes*, cut the density surface with the plane $x = 4$. The curve on the cut is *almost* the conditional distribution — you must still normalize it (its area is $f_X(4)$, not 1). The formula changes only its letters:

$$
f_{Y \mid X=x}(y) = \frac{f_{XY}(x, y)}{f_X(x)}
$$

— PMFs become PDFs, the $P$'s become $f$'s, and everything else is the same slice-and-normalize story.

## 3. Worked Examples

### Example 1: heights of the 9-year-olds

Slice: $(0, 0, 0, 0, 0.3, 0.1)$, row sum $0.4$ → conditional $(0, 0, 0, 0, \tfrac34, \tfrac14)$. Note how conditioning *sharpened* the distribution: marginally, height 49 had probability 0.3; among 9-year-olds it has 0.75.

### Example 2: the dice check

$$
P(Y = 1 \mid X = 4) = \frac{P_{XY}(4, 1)}{P_X(4)} = \frac{1/36}{1/6} = \frac16
$$

### Example 3: conditioning the sum on the first die

From the (first die, sum) band table: given $X = 3$, the possible sums are $4, \dots, 9$, each cell $\tfrac1{36}$, and $P_X(3) = \tfrac16$ — so $P(S = s \mid X = 3) = \tfrac16$ for $s = 4..9$: the sum is *uniform over six values* once the first die is known (it is just 3 + the second die).

### Example 4: the reverse conditional

Given the **sum is 7**, what is the first die? Column $S = 7$ of the same table has $\tfrac1{36}$ for every $x$, and $P_S(7) = \tfrac{6}{36}$, so $P(X = x \mid S = 7) = \tfrac16$ for all $x$ — sum 7 reveals nothing about the first die. (Try $S = 12$ instead: the conditional collapses to certainty, $X = 6$.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

ages = [7, 8, 9, 10]
heights = [45, 46, 47, 48, 49, 50]
joint = pd.DataFrame(
    [[0.1, 0.2, 0.0, 0.0, 0.0, 0.0],
     [0.0, 0.0, 0.2, 0.0, 0.0, 0.0],
     [0.0, 0.0, 0.0, 0.0, 0.3, 0.1],
     [0.0, 0.0, 0.0, 0.0, 0.0, 0.1]],
    index=pd.Index(ages, name="age"),
    columns=pd.Index(heights, name="height"))

### Slice and normalize: the children example

In [ ]:
row = joint.loc[9]                        # the slice: fix age = 9
conditional = row / row.sum()             # normalize by the row sum P_X(9)

print(f"slice at age 9:          {row.values}   (sums to {row.sum():.1f}, not 1)")
print(f"P_X(9) (the row sum):    {row.sum():.1f}")
print(f"conditional P(Y|X=9):    {conditional.values}   (sums to {conditional.sum():.1f})")
print(f"\nP(Y=49 | X=9) = 0.3/0.4 = {conditional[49]:.2f}")

### A general conditional function, tested on all rows

Every row, once normalized, is a genuine distribution — one conditional per value of the fixed variable.

In [ ]:
def conditional_of_col_given_row(joint, x):
    slice_ = joint.loc[x]
    return slice_ / slice_.sum()

for a in ages:
    c = conditional_of_col_given_row(joint, a)
    print(f"P(height | age={a:>2}) = {np.round(c.values, 3)}   sum = {c.sum():.1f}")

### The dice sanity check and the (die, sum) conditionals

In [ ]:
faces = np.arange(1, 7)
flat = pd.DataFrame(1 / 36, index=faces, columns=faces)

cond = conditional_of_col_given_row(flat, 4)
print(f"P(Y | X=4) for independent dice: {np.round(cond.values, 4)}")
print("Equal to the plain marginal 1/6 -- conditioning on an independent variable")
print("changes nothing.\n")

sums = np.arange(2, 13)
die_sum = pd.DataFrame(0.0, index=faces, columns=sums)     # rows: X, cols: S
for d1 in faces:
    for d2 in faces:
        die_sum.loc[d1, d1 + d2] += 1 / 36

cond_s_given_x = conditional_of_col_given_row(die_sum, 3)
print(f"P(S | X=3): nonzero at sums {list(cond_s_given_x[cond_s_given_x > 0].index)}"
      f" each = {cond_s_given_x.max():.4f}  (uniform: 3 + second die)")

col = die_sum[7]
cond_x_given_s7 = col / col.sum()
print(f"P(X | S=7)  = {np.round(cond_x_given_s7.values, 4)}  (sum 7 reveals nothing)")
col12 = die_sum[12]
print(f"P(X | S=12) = {np.round((col12 / col12.sum()).values, 4)}  (sum 12 forces X = 6)")

### Continuous: slicing the call-center density

Rebuild the empirical joint density, cut it at waiting time $x = 4$, and normalize the slice — the conditional PDF of the rating after a 4-minute wait.

In [ ]:
rng = np.random.default_rng(42)

n = 400_000
quick = rng.random(n) < 0.5
wait = np.clip(np.where(quick, rng.normal(2.4, 1.6, n), rng.normal(7.6, 1.6, n)), 0, 10)
rate = np.clip(np.where(quick, rng.normal(8.0, 1.7, n), rng.normal(2.6, 1.7, n)), 0, 10)

density, xe, ye = np.histogram2d(wait, rate, bins=50, range=[[0, 10], [0, 10]],
                                 density=True)
xc = (xe[:-1] + xe[1:]) / 2
yc = (ye[:-1] + ye[1:]) / 2

ix = np.argmin(np.abs(xc - 4.0))                  # the column closest to x = 4
slice_at_4 = density[ix]                          # f_XY(4, y): NOT a density yet
f_x_at_4 = np.trapezoid(slice_at_4, yc)           # its area = marginal f_X(4)
conditional_pdf = slice_at_4 / f_x_at_4           # normalize -> f_{Y|X=4}

print(f"area of the raw slice:        {f_x_at_4:.4f}   (= f_X(4), not 1)")
print(f"area after normalizing:       {np.trapezoid(conditional_pdf, yc):.4f}")
print(f"conditional mean rating after a 4-minute wait: "
      f"{np.trapezoid(yc * conditional_pdf, yc):.2f}")

## 5. Visualizations

### The slice in the table, before and after normalizing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8),
                         gridspec_kw={"width_ratios": [1.6, 1, 1]})

im = axes[0].imshow(joint.values, cmap="Blues", aspect="auto")
axes[0].add_patch(plt.Rectangle((-0.5, 1.5), 6, 1, fill=False,
                                edgecolor="tab:red", linewidth=3))
axes[0].set_xticks(range(6)); axes[0].set_xticklabels(heights)
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(ages)
axes[0].set_xlabel("height"); axes[0].set_ylabel("age")
axes[0].set_title("fix X = 9: keep only that row")

axes[1].bar(heights, joint.loc[9].values, color="tab:red", alpha=0.6,
            edgecolor="black")
axes[1].set_title("the raw slice (sums to 0.4)")
axes[1].set_xlabel("height")

axes[2].bar(heights, (joint.loc[9] / joint.loc[9].sum()).values,
            color="tab:green", alpha=0.7, edgecolor="black")
axes[2].set_title("normalized: P(Y | X = 9)")
axes[2].set_xlabel("height")

for ax in axes[1:]:
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("A conditional distribution is a slice of the joint, renormalized")
plt.tight_layout()
plt.show()

### Slicing the density mountain at x = 4

In [ ]:
fig = plt.figure(figsize=(13, 5))

ax0 = fig.add_subplot(1, 2, 1)
ax0.imshow(density.T, origin="lower", extent=[0, 10, 0, 10], cmap="viridis",
           aspect="auto")
ax0.axvline(4, color="tab:red", linewidth=2.5, label="slice at x = 4 min")
ax0.set_xlabel("waiting time (minutes)")
ax0.set_ylabel("rating")
ax0.set_title("cut the joint density with the plane x = 4")
ax0.legend()

ax1 = fig.add_subplot(1, 2, 2)
ax1.plot(yc, slice_at_4, color="gray", linewidth=2,
         label=f"raw slice $f_{{XY}}(4, y)$ (area {f_x_at_4:.2f})")
ax1.plot(yc, conditional_pdf, color="tab:red", linewidth=2.5,
         label="normalized: $f_{Y|X=4}(y)$ (area 1)")
ax1.set_xlabel("rating y")
ax1.set_ylabel("density")
ax1.set_title("the slice becomes the conditional PDF after normalizing")
ax1.grid(True, alpha=0.3)
ax1.legend()

plt.tight_layout()
plt.show()

### A family of conditionals: ratings for different waiting times

One conditional PDF per waiting time — watch the distribution of ratings collapse from happy to angry as the wait grows. This per-slice view is exactly what "the relationship between the variables" means.

In [ ]:
plt.figure(figsize=(10, 5.5))
for x0, color in [(1, "tab:green"), (4, "tab:olive"), (6, "tab:orange"),
                  (9, "tab:red")]:
    ix = np.argmin(np.abs(xc - x0))
    s = density[ix]
    cond = s / np.trapezoid(s, yc)
    plt.plot(yc, cond, color=color, linewidth=2.5,
             label=f"f(rating | wait = {x0} min)")
plt.xlabel("rating y")
plt.ylabel("conditional density")
plt.title("Conditional distributions of the rating, one per waiting time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Compute $P(Y \mid X = 7)$ for the children's table (slice, sum, normalize) — first by hand, then with code.

**Exercise 2.** Compute the reverse conditional $P(X \mid Y = 50)$: among the 50-inch children, what is the age distribution? (Slice a *column* this time.)

### Intermediate

**Exercise 3.** Verify computationally that for every age $x$, the conditional $P(Y \mid X = x)$ sums to 1, and that reassembling $P_X(x) \cdot P_{Y|X=x}(y)$ over all cells recovers the joint table exactly — the conditional-probability rule run backwards.

**Exercise 4.** For the (first die, sum) table, compute $P(S = 7 \mid X = x)$ for every $x$. Why is it the same for all $x$, and how does that connect to the fact that $P(X \mid S = 7)$ was uniform?

### Advanced

**Exercise 5.** Independence via conditionals: show numerically that for the two-dice table, $P(Y \mid X = x)$ equals the marginal $P_Y$ for *every* $x$ — and that for the children's table it does not. Explain why "all conditionals equal the marginal" is exactly the same statement as the factorization $P_{XY} = P_X P_Y$.

**Exercise 6.** Conditional means as a curve: for the call-center density, compute the conditional mean rating $E[Y \mid X = x]$ for every waiting-time bin and plot it against $x$. What familiar data-science object is this curve an example of? (Hint: it is what regression estimates.)

## 7. Solutions

In [ ]:
# Exercise 1: heights of the 7-year-olds
c7 = conditional_of_col_given_row(joint, 7)
print("Exercise 1: slice (0.1, 0.2, 0, 0, 0, 0), row sum 0.3")
print(f"P(Y | X=7) = {np.round(c7.values, 3)}")
print("One third of the 7-year-olds are 45 inches, two thirds are 46.")

**Exercise 1, illustrated.** A one-page visual summary of the solution above.

<img src="img/05_ex1_compute_conditional_probability.png" alt="Illustrated solution to Exercise 1" width="900">

In [ ]:
# Exercise 2: ages of the 50-inch children
col50 = joint[50]
cond_age = col50 / col50.sum()
print(f"Exercise 2: column 50 = {col50.values}, column sum {col50.sum():.1f}")
print(f"P(X | Y=50) = {np.round(cond_age.values, 2)}")
print("Half the 50-inch children are 9, half are 10.")

**Exercise 2, illustrated.** A one-page visual summary of the solution above.

<img src="img/05_ex2_compute_conditional_probability.png" alt="Illustrated solution to Exercise 2" width="900">

In [ ]:
# Exercise 3: conditionals sum to 1, and rebuild the joint
rebuilt = joint.copy() * 0.0
for a in ages:
    px = joint.loc[a].sum()
    cond = joint.loc[a] / px
    assert np.isclose(cond.sum(), 1.0)
    rebuilt.loc[a] = px * cond
print(f"Exercise 3: every conditional sums to 1 (asserted).")
print(f"P_X(x) * P(Y|X=x) rebuilds the joint exactly: "
      f"{np.allclose(rebuilt.values, joint.values)}")

**Exercise 3, illustrated.** A one-page visual summary of the solution above.

<img src="img/05_ex3_conditionals_sum_to_1.png" alt="Illustrated solution to Exercise 3" width="900">

In [ ]:
# Exercise 4: P(S=7 | X=x) for every x
print("Exercise 4:")
for x in faces:
    cond = die_sum.loc[x] / die_sum.loc[x].sum()
    print(f"  P(S=7 | X={x}) = {cond[7]:.4f}")
print("Always 1/6: whatever the first die shows, the second must land one exact")
print("value to make 7 -- and each value has probability 1/6. Because knowing X")
print("never changes the chance of S=7, knowing S=7 cannot change the")
print("distribution of X either: the two uniform conditionals are two views of")
print("the same fact.")

**Exercise 4, illustrated.** A one-page visual summary of the solution above.

<img src="img/05_ex4_symmetry_sum_given_first_die.png" alt="Illustrated solution to Exercise 4" width="900">

In [ ]:
# Exercise 5: independence <=> every conditional equals the marginal
marginal_y_dice = flat.sum(axis=0)
all_equal = all(np.allclose((flat.loc[x] / flat.loc[x].sum()).values,
                            marginal_y_dice.values) for x in faces)
print(f"Exercise 5: dice -- every P(Y|X=x) equals the marginal P_Y? {all_equal}")

marginal_h = joint.sum(axis=0)
c9 = conditional_of_col_given_row(joint, 9)
print(f"children -- P(Y|X=9) = {np.round(c9.values, 2)}")
print(f"            marginal = {np.round(marginal_h.values, 2)}  -> different!")
print()
print("P(Y|X=x) = P_Y for all x  <=>  P_XY(x,y) = P_X(x) P_Y(y) for all cells:")
print("multiply both sides of the conditional by P_X(x) and you get the")
print("factorization; divide the factorization by P_X(x) and you get the")
print("conditional. Independence = conditioning changes nothing.")

**Exercise 5, illustrated.** A one-page visual summary of the solution above.

<img src="img/05_ex5_independence_via_conditionals.png" alt="Illustrated solution to Exercise 5" width="900">

In [ ]:
# Exercise 6: the conditional mean curve E[Y | X=x]
cond_means = []
for i in range(len(xc)):
    s = density[i]
    area = np.trapezoid(s, yc)
    cond_means.append(np.trapezoid(yc * s, yc) / area if area > 0 else np.nan)

plt.figure(figsize=(9, 5))
plt.scatter(wait[:3000], rate[:3000], s=5, alpha=0.2, color="tab:blue",
            label="customers")
plt.plot(xc, cond_means, color="tab:red", linewidth=3,
         label="conditional mean  E[Y | X=x]")
plt.xlabel("waiting time (minutes)")
plt.ylabel("rating")
plt.title("The conditional mean traces the relationship through the cloud")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Exercise 6: the curve of conditional means IS the regression function --")
print("what regression models estimate is exactly E[Y | X=x] as a function of x.")

## 8. Conclusion

- A **conditional distribution** fixes one variable and describes the other: take the **slice** of the joint at $X = x$, then **normalize** by the slice's total (the marginal $P_X(x)$), because distributions must sum to 1.
- The formula — joint over marginal-of-the-fixed-variable — is the conditional-probability rule $P(A\,\text{and}\,B) = P(A)P(B \mid A)$ rearranged: $P_{Y|X=x}(y) = \frac{P_{XY}(x,y)}{P_X(x)}$; for the children, $P(Y{=}49 \mid X{=}9) = \frac{0.3}{0.4} = \frac34$.
- Independent dice pass the sanity check: every conditional equals the plain marginal ($\tfrac16$) — conditioning on an independent variable changes nothing, and that property *is* independence.
- Continuous version: slice the density surface and normalize the cut curve — $f_{Y|X=x}(y) = \frac{f_{XY}(x,y)}{f_X(x)}$, the same formula with $P$'s replaced by $f$'s.
- Conditionals are the joint's "views": the family of slices shows how one variable's distribution *moves* with the other (ratings collapsing as waits grow), and the curve of conditional means is precisely what regression estimates.

Next: distilling all of this dependence into single numbers — **covariance** and correlation.